In [0]:
import json
import os
import time

from pathlib import Path
from typing import Any, Mapping

import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


class FootballDataOrgError(RuntimeError):
    """Base exception for football-data.org client errors."""


class FootballDataOrgRateLimitError(FootballDataOrgError):
    """Raised when football-data.org rejects a request due to rate limits."""


class FootballDataOrgClient:
    """
    football-data.org HTTP client.

    Features:
      - persistent HTTP session
      - retry/backoff for temporary server failures
      - configurable request pacing
      - HTTP and API-level error handling
      - rate-limit tracking
      - context-manager support
    """

    def __init__(
        self,
        api_token: str,
        base_url: str = "https://api.football-data.org/v4",
        min_request_interval_seconds: float = 6.5,
        timeout_seconds: int = 30,
    ) -> None:

        if not api_token:
            raise ValueError("api_token cannot be empty.")

        if not base_url:
            raise ValueError("base_url cannot be empty.")

        if min_request_interval_seconds < 0:
            raise ValueError(
                "min_request_interval_seconds cannot be negative."
            )

        if timeout_seconds <= 0:
            raise ValueError(
                "timeout_seconds must be greater than zero."
            )

        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.min_request_interval_seconds = (
            min_request_interval_seconds
        )

        self._headers = {
            "X-Auth-Token": api_token,
            "Accept": "application/json",
        }

        self._last_request_started_at: float | None = None

        self.requests_available_minute: int | None = None
        self.request_counter_reset_seconds: int | None = None

        retry = Retry(
            total=3,
            connect=3,
            read=3,
            status=3,
            backoff_factor=1.5,
            status_forcelist=(
                500,
                502,
                503,
                504,
            ),
            allowed_methods=frozenset({"GET"}),
            respect_retry_after_header=True,
        )

        adapter = HTTPAdapter(
            max_retries=retry,
        )

        self.session = requests.Session()

        self.session.mount(
            "https://",
            adapter,
        )

    def __enter__(self) -> "FootballDataOrgClient":
        return self

    def __exit__(
        self,
        exc_type,
        exc_value,
        traceback,
    ) -> None:
        self.close()

    def _wait_for_request_slot(self) -> None:
        """Respect the configured delay between requests."""

        if self._last_request_started_at is None:
            return

        elapsed = (
            time.monotonic()
            - self._last_request_started_at
        )

        remaining_wait = (
            self.min_request_interval_seconds
            - elapsed
        )

        if remaining_wait > 0:
            time.sleep(remaining_wait)

    @staticmethod
    def _to_int(
        value: str | None,
    ) -> int | None:
        """Safely convert an HTTP response header to integer."""

        if value is None:
            return None

        try:
            return int(value)
        except (TypeError, ValueError):
            return None

    def _update_rate_limit_state(
        self,
        response: requests.Response,
    ) -> None:
        """Update known API rate-limit state from response headers."""

        self.requests_available_minute = self._to_int(
            response.headers.get(
                "X-Requests-Available-Minute"
            )
        )

        self.request_counter_reset_seconds = self._to_int(
            response.headers.get(
                "X-RequestCounter-Reset"
            )
        )

    def _validate_known_quota(self) -> None:
        """
        Fail early when the previously observed per-minute
        request quota is exhausted.
        """

        if (
            self.requests_available_minute is not None
            and self.requests_available_minute <= 0
        ):
            reset_seconds = (
                self.request_counter_reset_seconds
                if self.request_counter_reset_seconds is not None
                else "unknown"
            )

            raise FootballDataOrgRateLimitError(
                "football-data.org per-minute API quota is exhausted. "
                f"Counter reset in approximately {reset_seconds} seconds."
            )

    def get(
        self,
        endpoint: str,
        params: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        """
        Execute one GET request and return the validated JSON payload.
        """

        if not endpoint:
            raise ValueError("endpoint cannot be empty.")

        self._validate_known_quota()
        self._wait_for_request_slot()

        url = (
            f"{self.base_url}/"
            f"{endpoint.lstrip('/')}"
        )

        self._last_request_started_at = (
            time.monotonic()
        )

        try:
            response = self.session.get(
                url=url,
                params=dict(params) if params else None,
                headers=self._headers,
                timeout=self.timeout_seconds,
            )

        except requests.RequestException as exc:
            raise FootballDataOrgError(
                "football-data.org request failed for "
                f"endpoint '{endpoint}'."
            ) from exc

        self._update_rate_limit_state(response)

        if response.status_code == 429:
            reset_seconds = (
                response.headers.get(
                    "X-RequestCounter-Reset"
                )
            )

            message = (
                "football-data.org returned HTTP 429 "
                "(rate limit exceeded)."
            )

            if reset_seconds:
                message += (
                    f" Request counter resets in approximately "
                    f"{reset_seconds} seconds."
                )

            raise FootballDataOrgRateLimitError(
                message
            )

        try:
            response.raise_for_status()

        except requests.HTTPError as exc:

            error_message = None

            try:
                error_payload = response.json()

                if isinstance(error_payload, dict):
                    error_message = error_payload.get(
                        "message"
                    ) or error_payload.get(
                        "error"
                    )

            except ValueError:
                pass

            message = (
                "football-data.org returned HTTP "
                f"{response.status_code} "
                f"for endpoint '{endpoint}'."
            )

            if error_message:
                message += f" API message: {error_message}"

            raise FootballDataOrgError(
                message
            ) from exc

        try:
            payload = response.json()

        except ValueError as exc:
            raise FootballDataOrgError(
                "football-data.org returned a non-JSON "
                f"response for endpoint '{endpoint}'."
            ) from exc

        if not isinstance(payload, dict):
            raise FootballDataOrgError(
                "football-data.org returned an unexpected "
                f"JSON structure for endpoint '{endpoint}'."
            )

        if payload.get("error"):
            raise FootballDataOrgError(
                "football-data.org returned an API-level "
                f"error for endpoint '{endpoint}': "
                f"{payload['error']}"
            )

        return payload

    def get_competition(
        self,
        competition_code: str,
    ) -> dict[str, Any]:
        """
        Extract current competition metadata.

        Existing raw JSON files are replaced atomically.
        This keeps currentSeason and related metadata up to date.
        """

        if not competition_code:
            raise ValueError(
                "competition_code cannot be empty."
            )

        return self.get(
            endpoint=f"competitions/{competition_code}",
        )

    def get_competition_teams(
        self,
        competition_code: str,
    ) -> dict[str, Any]:
        """
        Extract teams for the API's default current season.

        Existing raw files are replaced atomically.
        """

        if not competition_code:
            raise ValueError(
                "competition_code cannot be empty."
            )

        return self.get(
            endpoint=f"competitions/{competition_code}/teams",
        )

    def get_competition_matches(
        self,
        competition_code: str,
    ) -> dict[str, Any]:
        """
        Extract the API's currently available season.

        Each competition response is stored as one raw JSON file.

        Existing raw files are replaced atomically so changes to match
        statuses, scores, and fixtures can reach Bronze.
        """

        if not competition_code:
            raise ValueError(
                "competition_code cannot be empty."
            )

        return self.get(
            endpoint=f"competitions/{competition_code}/matches",
        )

    def get_competition_standings(
        self,
        competition_code: str,
    ) -> dict[str, Any]:
        """
        Extract current standings for configured competitions.

        Each execution refreshes the stored standings snapshot.
        Existing raw files are replaced atomically.
        """

        if not competition_code:
            raise ValueError(
                "competition_code cannot be empty."
            )

        return self.get(
            endpoint=f"competitions/{competition_code}/standings",
        )

    def close(self) -> None:
        """Close the underlying HTTP session."""

        self.session.close()


def write_json_atomic(
    data: Any,
    file_path: str,
) -> None:
    """
    Write JSON through a temporary file.

    The final path is replaced only after serialization succeeds,
    preventing partially written final output.
    """

    path = Path(file_path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    try:
        with temporary_path.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                data,
                file,
                indent=2,
                ensure_ascii=False,
            )

        os.replace(
            temporary_path,
            path,
        )

    except Exception:

        if temporary_path.exists():
            temporary_path.unlink()

        raise